# Импорт библиотек, загрузка датасета и его осмотр.

In [15]:
import pandas as pd
import numpy as np
import matplotlib as mlp

In [16]:
df = pd.read_csv('/content/drive/MyDrive/datasets/topic_small.csv') # датасет в гугл драйве

In [17]:
df.loc[df['topic'].isna(),'topic'].sum() #есть ли наны в целевой

0

In [18]:
df.head()

,Unnamed: 0,url,title,text,topic,tags,date
0,390718,https://lenta.ru/news/2011/09/26/cost1/,Расходы Великобритании на операцию в Ливии нед...,Расходы правительства Великобритании на военну...,Мир,Все,2011/09/26
1,661864,https://lenta.ru/news/2017/05/19/arthouse/,Мединский предрек превращение российского кино...,Если не оказывать отечественному кинематографу...,Культпросвет,События,2017/05/19
2,139745,https://lenta.ru/news/2005/12/27/offer/,Украина предложила 80 долларов за тысячу кубом...,Украина готова платить 80 долларов за тысячу к...,Бывший СССР,Все,2005/12/27
3,655163,https://lenta.ru/news/2017/04/08/badrussian/,Маккейн назвал Россию и президента Сирии «один...,Член сената Конгресса США республиканец Джон М...,Мир,Политика,2017/04/08
4,338861,https://lenta.ru/news/2010/07/23/mansion/,Гильермо дель Торо сделает фильм из диснеевско...,Гильермо дель Торо сделает фильм из диснеевско...,Культура,Все,2010/07/23


In [19]:
df.shape

(100000, 7)

Сколько классов и какие из них можно было бы убрать

In [20]:
df['topic'].value_counts()

,count
topic,
Россия,20082
Мир,17120
Экономика,9929
Спорт,8042
Наука и техника,6665
Бывший СССР,6601
Культура,6568
Интернет и СМИ,5563
Из жизни,3438


Все, что меньше путешествий по количеству можно убрать, так как их слишком мало, смысла особо для модели не имеют

# Корректировка датасета

## Обрезание малоинформативных топиков

In [21]:
topics = df['topic'].value_counts()

In [22]:
topics_yes = topics[topics>=500].index # оставляем топики где количество статей больше 500

In [23]:
topics_yes

Index(['Россия', 'Мир', 'Экономика', 'Спорт', 'Наука и техника', 'Бывший СССР',
       'Культура', 'Интернет и СМИ', 'Из жизни', 'Дом', 'Силовые структуры',
       'Ценности', 'Бизнес', 'Путешествия'],
      dtype='object', name='topic')

In [24]:
df = df[df['topic'].isin(topics_yes)].reset_index(drop=True)

In [25]:
df['topic'].value_counts()

,count
topic,
Россия,20082
Мир,17120
Экономика,9929
Спорт,8042
Наука и техника,6665
Бывший СССР,6601
Культура,6568
Интернет и СМИ,5563
Из жизни,3438


In [26]:
df.shape

(91957, 7)

## Обработка текстов статей

In [27]:
df['text'].iloc[0] # смотрим как выглядит текст одной из статей, чтобы понять как обработать

'Расходы правительства Великобритании на военную операцию в Ливии могут достигнуть 1,75 миллиарда фунтов, сообщает The Guardian со ссылкой на эксперта авторитетного издания Defence Analysis. Согласно расчетам эксперта Фрэнсиса Тусы (Francis Tusa), правительство могло в семь раз недооценить возможные расходы на участие военно-воздушных сил в бомбардировках сил Муаммара Каддафи. Для расчета расходов на бомбардировки Ливии Туса опирался на данные, предоставленные членами британского парламента и представителями военно-воздушных сил. При своих подсчетах он использовал две разные методики: в первом случае полученный результат колебался между 1,38 и 1,58 миллиарда фунтов, во втором случае разброс оказался еще больше - от 850 миллионов до 1,75 миллиарда фунтов. При этом Туса подчеркнул, что в своих расчетах он не учитывал последние вылеты королевских ВВС в Ливии в сентябре. Таким образом, по окончании операции итоговая сумма может существенно вырасти. Для сравнения, США потратили на операцию 

In [28]:
import re

In [29]:
def text_clean(text):
  text = str(text).lower() # нижний регистр
  # text = re.sub(r'https:\S+|www\S+','URLTOKEN',text)
  text = re.sub(r'\d+',' numtoken ',text)
  text = re.sub(r'[^а-яёa-z\s]',' ',text)# убираем все другие символы кроме кириллицы и латиницы
  text = re.sub(r'\s+',' ',text)
  return text

In [30]:
df['text_clean'] = df['text'].apply(text_clean)

In [31]:
df['text_clean'].iloc[0]

'расходы правительства великобритании на военную операцию в ливии могут достигнуть numtoken numtoken миллиарда фунтов сообщает the guardian со ссылкой на эксперта авторитетного издания defence analysis согласно расчетам эксперта фрэнсиса тусы francis tusa правительство могло в семь раз недооценить возможные расходы на участие военно воздушных сил в бомбардировках сил муаммара каддафи для расчета расходов на бомбардировки ливии туса опирался на данные предоставленные членами британского парламента и представителями военно воздушных сил при своих подсчетах он использовал две разные методики в первом случае полученный результат колебался между numtoken numtoken и numtoken numtoken миллиарда фунтов во втором случае разброс оказался еще больше от numtoken миллионов до numtoken numtoken миллиарда фунтов при этом туса подчеркнул что в своих расчетах он не учитывал последние вылеты королевских ввс в ливии в сентябре таким образом по окончании операции итоговая сумма может существенно вырасти д

# Лемматизация текста

In [32]:
!pip install pymorphy3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 76.6 MB/s eta 0:00:00


In [33]:
import pymorphy3

In [34]:
from collections import Counter
all_words = ' '.join(df['text_clean']).split() # все слова
unique_words = set(all_words)# уникальные
print(len(unique_words))  # вывод сколько уникальных слов

408206


In [35]:
morph = pymorphy3.MorphAnalyzer()
def lemmatize(text):
    words = text.split()
    lemmas = [morph.parse(word)[0].normal_form for word in words]
    return ' '.join(lemmas)
# лемматизация без кэширования (словарик уникальных слов не нужен)

In [36]:
lemma_dict = {word: morph.parse(word)[0].normal_form for word in unique_words}
def lemmatize_cache(text):
  return ' '.join(lemma_dict.get(word,word) for word in text.split())
df['text_lemma'] = df['text_clean'].apply(lemmatize_cache)
# лемматизация с кэшированием

In [37]:
df.loc[df['topic'].isna(),['topic','text_lemma']]

,topic,text_lemma


In [38]:
df.shape

(91957, 9)

# Деление выборки и модель

In [39]:
from sklearn.model_selection import train_test_split

In [40]:
X = df['text_lemma']
y = df['topic']
X_train,X_val,y_train,y_val = train_test_split(X,y,test_size=0.1,random_state=42,stratify=y)

## Векторизация

In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=20000, min_df=3)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

In [42]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=500)
model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=500)

## Метрики

In [43]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_val_tfidf)
print(classification_report(y_val, y_pred))

                   precision    recall  f1-score   support

           Бизнес       0.68      0.18      0.29        93
      Бывший СССР       0.83      0.82      0.82       660
              Дом       0.87      0.75      0.81       273
         Из жизни       0.70      0.62      0.65       344
   Интернет и СМИ       0.79      0.73      0.76       556
         Культура       0.87      0.88      0.87       657
              Мир       0.79      0.84      0.82      1712
  Наука и техника       0.83      0.87      0.85       667
      Путешествия       0.90      0.54      0.68        85
           Россия       0.77      0.83      0.80      2008
Силовые структуры       0.70      0.41      0.52       250
            Спорт       0.96      0.97      0.97       804
         Ценности       1.00      0.74      0.85        94
        Экономика       0.83      0.86      0.85       993

         accuracy                           0.82      9196
        macro avg       0.82      0.72      0.75      

Здесь сделал сбалансированную модель, то есть она будет уделять больше внимания более редким топикам

In [44]:
model_balanced = LogisticRegression(max_iter=1000, n_jobs=-1, class_weight='balanced')
model_balanced.fit(X_train_tfidf, y_train)

y_pred_balanced = model_balanced.predict(X_val_tfidf)
print(classification_report(y_val, y_pred_balanced))

                   precision    recall  f1-score   support

           Бизнес       0.38      0.69      0.49        93
      Бывший СССР       0.79      0.88      0.83       660
              Дом       0.71      0.85      0.78       273
         Из жизни       0.56      0.77      0.65       344
   Интернет и СМИ       0.75      0.77      0.76       556
         Культура       0.88      0.88      0.88       657
              Мир       0.84      0.78      0.81      1712
  Наука и техника       0.82      0.85      0.84       667
      Путешествия       0.55      0.85      0.66        85
           Россия       0.86      0.69      0.77      2008
Силовые структуры       0.46      0.75      0.57       250
            Спорт       0.96      0.97      0.97       804
         Ценности       0.87      0.89      0.88        94
        Экономика       0.87      0.82      0.84       993

         accuracy                           0.80      9196
        macro avg       0.74      0.82      0.77      

# Дообучение модели ruBERT-tiny2 для классификации

In [45]:
df_bert = df[['text','topic']].copy()

In [46]:
df_sample, _ = train_test_split(df_bert, train_size = 18000, stratify = df_bert['topic'],random_state = 42)
df_train,df_val = train_test_split(df_sample,test_size=0.1,stratify=df_sample['topic'],random_state= 42)
print(df_train.shape, ' ',df_val.shape)

(16200, 2)   (1800, 2)


In [47]:
df_sample['topic'].value_counts() #смотрим сохранились ли отношения классов

,count
topic,
Россия,3931
Мир,3351
Экономика,1943
Спорт,1574
Наука и техника,1305
Бывший СССР,1292
Культура,1286
Интернет и СМИ,1089
Из жизни,673


In [48]:
from sklearn.preprocessing import LabelEncoder

## Кодировка

In [49]:
enc = LabelEncoder() # кодирование категориальных данных
df_train['label'] = enc.fit_transform(df_train['topic'])
df_val['label'] = enc.transform(df_val['topic'])

In [50]:
enc.classes_

array(['Бизнес', 'Бывший СССР', 'Дом', 'Из жизни', 'Интернет и СМИ',
       'Культура', 'Мир', 'Наука и техника', 'Путешествия', 'Россия',
       'Силовые структуры', 'Спорт', 'Ценности', 'Экономика'],
      dtype=object)

## Трансформер

In [51]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [52]:
model_name = 'cointegrated/rubert-tiny2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_bert = AutoModelForSequenceClassification.from_pretrained(model_name,num_labels = len(enc.classes_))

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  118MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [53]:
df_train = df_train.dropna(subset=['text'])

### Создание необходимых функций

In [54]:
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=256) #функция токенизации

from datasets import Dataset

train_dataset = Dataset.from_pandas(df_train[['text', 'label']])
val_dataset = Dataset.from_pandas(df_val[['text', 'label']])

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/16199 [00:00<?, ? examples/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

In [55]:
from sklearn.metrics import accuracy_score,f1_score

In [56]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=1)
  acc = accuracy_score(labels,predictions)
  f1_macro = f1_score(labels,predictions,average = 'macro')
  f1_weighted = f1_score(labels,predictions,average = 'weighted')
  return{'accuracy =':acc, 'f1 score(macro) =':f1_macro,'f1 score(weighted) =':f1_weighted
  } #функция вычисления метрик

In [57]:
from transformers import TrainingArguments, Trainer

In [58]:
training_args = TrainingArguments(output_dir='./bert_results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1 score(macro) =',
    logging_steps=50,
    report_to='none'
)

In [59]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # использование серверов колаба

In [60]:
model_bert.to(device)# переключаем модель на колаб

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(83828, 312, padding_idx=0)
      (position_embeddings): Embedding(2048, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-2): 3 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=312, out_features=312, bias=True)
              (key): Linear(in_features=312, out_features=312, bias=True)
              (value): Linear(in_features=312, out_features=312, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=312, out_features=312, bias=True)
              (LayerNorm): LayerNorm((312,), eps=1e-12, 

In [61]:
trainer = Trainer(model = model_bert,args = training_args,train_dataset=train_dataset,eval_dataset=val_dataset,compute_metrics=compute_metrics)
trainer.train() # обучение

Epoch,Training Loss,Validation Loss,Accuracy =,F1 score(macro) =,F1 score(weighted) =
1,1.079330,1.055170,0.715556,0.431198,0.667180
2,0.872271,0.835030,0.765556,0.548698,0.737988
3,0.716857,0.757441,0.775556,0.578804,0.753424
4,0.620207,0.728241,0.776667,0.599089,0.757643
5,0.598356,0.717300,0.781111,0.614226,0.763386


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5065, training_loss=0.8701831768672299, metrics={'train_runtime': 284.3196, 'train_samples_per_second': 284.873, 'train_steps_per_second': 17.814, 'total_flos': 299104224660480.0, 'train_loss': 0.8701831768672299, 'epoch': 5.0})

In [62]:
rez = trainer.evaluate() # лучший результат

Training Loss,Validation Loss,Epoch,Accuracy =,F1 score(macro) =,F1 score(weighted) =
0.598356,0.717300,5,0.781111,0.614226,0.763386


## Результаты классического и глубокого подходов на тренировочных данных

In [63]:
print(classification_report(y_val, y_pred_balanced))#Классика

                   precision    recall  f1-score   support

           Бизнес       0.38      0.69      0.49        93
      Бывший СССР       0.79      0.88      0.83       660
              Дом       0.71      0.85      0.78       273
         Из жизни       0.56      0.77      0.65       344
   Интернет и СМИ       0.75      0.77      0.76       556
         Культура       0.88      0.88      0.88       657
              Мир       0.84      0.78      0.81      1712
  Наука и техника       0.82      0.85      0.84       667
      Путешествия       0.55      0.85      0.66        85
           Россия       0.86      0.69      0.77      2008
Силовые структуры       0.46      0.75      0.57       250
            Спорт       0.96      0.97      0.97       804
         Ценности       0.87      0.89      0.88        94
        Экономика       0.87      0.82      0.84       993

         accuracy                           0.80      9196
        macro avg       0.74      0.82      0.77      

In [64]:
print(rez)#Трансформер

{'eval_loss': 0.7172996401786804, 'eval_accuracy =': 0.7811111111111111, 'eval_f1 score(macro) =': 0.6142257924398418, 'eval_f1 score(weighted) =': 0.7633855989722829}


# Дообученная модель и классика на новых данных

## Загрузка полного датасета, из которого был получен тренировочный датасет

In [65]:
testdf = pd.read_csv('/content/drive/MyDrive/datasets/lenta-ru-news.csv.bz2',compression = 'bz2') #тестовый датасет загружается из гугл драйва

/tmp/ipykernel_1289/4057257612.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  testdf = pd.read_csv('/content/drive/MyDrive/datasets/lenta-ru-news.csv.bz2',compression = 'bz2') #тестовый датасет загружается из гугл драйва


In [66]:
testdf.describe()

,url,title,text,topic,tags,date
count,800975,800975,800970,738973,773756,800975
unique,800964,797832,800037,23,94,7393
top,https://lenta.ru/news/2004/06/21/hostage/,В Москве объявлено штормовое предупреждение,"РИА ""Новости""",Россия,Все,2019/12/05
freq,2,21,291,160445,453762,284


In [67]:
testdf['topic'].value_counts().sum()

np.int64(738973)

In [68]:
testdf = testdf[testdf['topic'].isin(topics_yes)].reset_index(drop=True)#оставляем те же топики что и в тренировке

In [69]:
testdf = testdf.sample(100000,random_state = 42).reset_index(drop=True) #Обрезаем

In [70]:
testdf = testdf.dropna(subset=['topic','text'])

In [71]:
testdf['topic'].unique()

array(['Экономика', 'Россия', 'Силовые структуры', 'Культура', 'Мир',
       'Из жизни', 'Интернет и СМИ', 'Наука и техника', 'Спорт',
       'Ценности', 'Бывший СССР', 'Путешествия', 'Дом', 'Бизнес'],
      dtype=object)

In [72]:
testdf['topic'].isna().sum()

np.int64(0)

## Предобработка текстов, так же как и на тренировочных

In [73]:
testdf['text_clean'] = testdf['text'].apply(text_clean)

In [74]:
new_all_words = ' '.join(testdf['text_clean']).split() # все слова нового датасета
new_unique_words = set(new_all_words)# уникальные
print(len(unique_words))

408206


In [75]:
new_lemma_dict = {word: morph.parse(word)[0].normal_form for word in new_unique_words}
def new_lemmatize_cache(text):
  return ' '.join(new_lemma_dict.get(word,word) for word in text.split())

In [76]:
testdf['text_lemma'] = testdf['text_clean'].apply(new_lemmatize_cache)

## Векторизация и результат логистической регрессии

In [77]:
X_new = vectorizer.transform(testdf['text_lemma'])
testdf['predicted_topic'] = model.predict(X_new)

In [78]:
print(classification_report(testdf['topic'],testdf['predicted_topic']))

                   precision    recall  f1-score   support

           Бизнес       0.74      0.23      0.35       953
      Бывший СССР       0.84      0.83      0.83      7292
              Дом       0.87      0.78      0.82      2973
         Из жизни       0.70      0.59      0.64      3809
   Интернет и СМИ       0.79      0.73      0.76      6039
         Культура       0.88      0.89      0.89      7358
              Мир       0.80      0.85      0.83     18499
  Наука и техника       0.83      0.86      0.84      7171
      Путешествия       0.83      0.57      0.68       837
           Россия       0.79      0.85      0.82     21887
Силовые структуры       0.75      0.41      0.53      2669
            Спорт       0.96      0.96      0.96      8665
         Ценности       0.95      0.74      0.83      1054
        Экономика       0.83      0.87      0.85     10794

         accuracy                           0.82    100000
        macro avg       0.83      0.73      0.76    10

## Применение дообученой модели(через pipeline) и результат

In [79]:
from transformers import pipeline

In [80]:
id2label = {i: cls for i, cls in enumerate(enc.classes_)}
label2id = {cls: i for i, cls in enumerate(enc.classes_)}

model_bert.config.id2label = id2label
model_bert.config.label2id = label2id

In [81]:
classifier = pipeline('text-classification', model=model_bert, tokenizer=tokenizer,
                       device=0 if torch.cuda.is_available() else -1)

In [82]:
results = classifier(testdf['text'].tolist(), truncation=True, max_length=256, batch_size=32)
testdf['predicted_topic'] = [r['label'] for r in results]

In [83]:
print(classification_report(testdf['topic'],testdf['predicted_topic']))

                   precision    recall  f1-score   support

           Бизнес       0.40      0.00      0.00       953
      Бывший СССР       0.78      0.86      0.82      7292
              Дом       0.70      0.74      0.72      2973
         Из жизни       0.61      0.43      0.51      3809
   Интернет и СМИ       0.72      0.72      0.72      6039
         Культура       0.81      0.91      0.85      7358
              Мир       0.80      0.82      0.81     18499
  Наука и техника       0.76      0.84      0.80      7171
      Путешествия       0.92      0.03      0.05       837
           Россия       0.77      0.82      0.79     21887
Силовые структуры       0.56      0.06      0.11      2669
            Спорт       0.94      0.97      0.96      8665
         Ценности       0.83      0.64      0.72      1054
        Экономика       0.79      0.86      0.82     10794

         accuracy                           0.79    100000
        macro avg       0.74      0.62      0.62    10

# Итоги

Посмотрим на результаты. В целом что на тренировочных что на тестовых данных ситуации схожи. Классика более точно классифицирует. Произошло это из-за маленького объема тренировочных данных для трансформера.

В целом, исправив датасеты и научив на них модель, можно получить значительно лучший результат. Также стоит учитывать, что я использовал модель с маленьким числом параметров(rubert-tiny2) и обрезал выборку обучения для берта, это тоже повлияло на результат.

Итог таков: в этой ситуации классический подход оказался лучше чем нейросетевой подход. Классика быстрее, проще в написании кода и архитектуре в целом, меньше по ресурсным затратам. Однако утверждать, что классика точнее не совсем корректно, так как изначальные данные не совсем равны как я описал выше, но и суть сравнения не в точности(нейросетевой подход при остальных равных зачастую оказывается точнее), а в ответе на вопрос: стоит ли точность всех затрат ресурсов и времени.